# [Solar ZED](https://www.kaggle.com/c/solar-zed)
Michał Jabłoński, 2019.01.23

## Cel zadania
Celem zadania było przeprowadzenie estymacji ilości wytworzonej energii przez panele słoneczne. Do jej wykonania użyto bibliotek `sklearn` oraz algorytmu RandomFroest (lasów losowych).

## Dane
Do wykonania estymacji otrzymujemy dwa pliki danych w formacie `*.csv`. Dzięki temu, że dane zostały podzielone na dane tesowe i treningowe, możliwe było zastosowanie regresji. Odczyt plików csv został wykonany z użyciem biblioteki `pandas`. Następnie zostało wykonane oczyszczanie pliku z wartości NA. Pole daty było zakodowane w formacie tekstowym `%m/%d/%Y %H:%M`, aby móc zastosować regresję, wymagena była zamiana na formę liczbową. Do tego została użyta biblioteka `datetime` i funkcja [strptime](https://docs.python.org/3/library/datetime.html), za pomocą której mozna było dokonać konwersji do wartości liczbowych. Jednak funkcja ta nie mogła zostać użyta na całym data frame'ie - należało przeiterować wszystkie wiersze manualnie, lub wykożystać funkcję [apply](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.apply.html) wraz z funkcją lambdą. W tym procesie utworzone zostały kolumny `year`, `month`, `day`, `hour`, bez minut, gdyż każdy wiersz daty zawierał datę z równą godziną (z 0 minut). Pod sam koniec przetwarzania danych usunięta została zbędna kolumna `data` Proces *czyszczenia* danych został zamknięty w funkcję, aby nie duplikować kodu dla odczytu danych z pliku `train.csv` oraz `test.csv`.

In [15]:
import pandas as pd
from datetime import datetime as dt

In [16]:
def prepareData(file_name):
    csv = pd.read_csv(file_name)
    csv.dropna()
        
    date_format = '%m/%d/%Y %H:%M' 

    csv['year'] = csv['data'].apply(lambda x: dt.strptime(x, date_format).year)
    csv['month'] = csv['data'].apply(lambda x: dt.strptime(x, date_format).month)
    csv['day'] = csv['data'].apply(lambda x: dt.strptime(x, date_format).day)
    csv['hour'] = csv['data'].apply(lambda x: dt.strptime(x, date_format).hour)

    csv.drop("data", axis = 1, inplace = True)
    return csv

In [17]:
train_data = prepareData('train.csv')
test_data = prepareData('test.csv')

In [18]:
train_data.head()

,id,idsito,idmodel,idbrand,lat,lon,ageinmonths,anno,day,ora,...,pcnm12,pcnm13,pcnm14,pcnm15,irr_pvgis_mod,irri_pvgis_mod,kwh,year,month,hour
0,97595,0.425,0.083,0.083,0.437,0.631,1.000,2012,2,0.0,...,0.828,0.562,0.485,0.612,0.0,0.189,0.0,2012,1,2
1,97606,0.250,0.208,0.083,0.437,0.620,0.063,2012,2,0.0,...,0.990,0.841,0.606,0.616,0.0,0.200,0.0,2012,1,2
2,97637,0.350,0.375,0.000,0.439,0.626,0.000,2012,2,0.0,...,0.884,0.597,0.539,0.614,0.0,0.158,0.0,2012,1,2
3,97646,0.325,0.292,0.167,0.440,0.612,0.250,2012,2,0.0,...,0.787,1.000,0.432,0.620,0.0,0.204,0.0,2012,1,2
4,97772,0.000,0.000,0.333,0.436,0.626,0.719,2012,2,0.0,...,0.748,0.614,0.473,0.614,0.0,0.199,0.0,2012,1,2


In [19]:
test_data.head()

,id,idsito,idmodel,idbrand,lat,lon,ageinmonths,anno,day,ora,...,pcnm11,pcnm12,pcnm13,pcnm14,pcnm15,irr_pvgis_mod,irri_pvgis_mod,year,month,hour
0,75823,0.150,0.167,0.167,0.438,0.630,0.0,2013,1,0.556,...,0.424,1.000,0.137,0.405,0.613,0.648,0.158,2013,9,12
1,75825,0.275,0.167,0.167,0.437,0.622,0.0,2013,30,0.556,...,1.000,0.972,0.738,0.644,0.615,0.587,0.281,2013,8,12
2,75858,0.275,0.167,0.167,0.437,0.622,0.0,2013,3,0.556,...,1.000,0.972,0.738,0.644,0.615,0.587,0.281,2013,9,12
3,75860,0.275,0.167,0.167,0.437,0.622,0.0,2013,4,0.556,...,1.000,0.972,0.738,0.644,0.615,0.587,0.281,2013,9,12
4,75906,0.150,0.167,0.167,0.438,0.630,0.0,2013,10,0.556,...,0.424,1.000,0.137,0.405,0.613,0.648,0.158,2013,9,12


## Podzbiór testowy
W celu przewidzenia regresji, [wydzielony został zbiór treningowy oraz testowy](https://towardsdatascience.com/train-test-split-and-cross-validation-in-python-80b61beca4b6), aby wytrenować model dla uzyskania jak najlepszej wartości `rmse`. Do podziału zbioru danych wykożystana została biblioteka `sklearn.model_selection` i funkcja [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html). Dmoyślne próbka danych to 0.25. Z danych znajduących się na osi `X` usunięta została kolumna kwh, która znalazła się na osi `Y`.

In [20]:
from sklearn.model_selection import train_test_split as tts

x = train_data.drop("kwh", 1)
y = train_data.loc[:,'kwh']

x_train, x_test, y_train, y_test = tts(x, y, random_state=0)

## Uczenie
Regresja została wykonana za pomocą [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html), na podstawie wcześniej wydzielonego zbioru danych testowych/treningowych. Do obliczenia błędu `rmse` wykożystano [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html). Obie funkcje pochodzą z pakietu `sklearn`. Ilość *rozgałęzień* określiłem na 500, co dało bardzo przyzwoite wyniki przy dużej szybkości wykonania, za róowno dla współczynika `coefficient` jak i `rmse`.

In [21]:
from sklearn.ensemble import RandomForestRegressor as rfr
from sklearn.metrics import mean_squared_error as mse

regressor = rfr(n_estimators = 500)
regressor = regressor.fit(x_train, y_train)

y_predicted = regressor.predict(x_test)

coefficient = regressor.score(x_train, y_train)
rmse = mse(y_test, y_predicted)

'rmse = ' + str(rmse) + ' coefficient = ' +  str(coefficient)

'rmse = 0.002262261220143528 coefficient = 0.9927889970696105'

In [22]:
y_predicted = regressor.predict(test_data)
predicted_df = pd.DataFrame(data={'Id': test_data['id'], 'Predicted': y_predicted})

# Wartości ujemne
Panele słoneczne nie mogą zabierać energii, tylko ją wytwarzać (pominięte zostały czynniki pomijalne w tej estymacji, np. zasilanie sterownika panelu słunecznego), ale jako, że użyliśmy `RandomForestRegressor` - wartości ujemne nie powstały. W przypadku użycia regresji liniowej musielibyśmy z takim wartościami sobie poradzić, np. zamieniając je na zera.

# Wyniki
Wyniki zostały zapisane w pliku `submission.csv`, w którym znalazły się dwie wartości - `Id` oraz `Predicted`.

In [24]:
predicted_df.to_csv('submission.csv', index = False, sep=',')